In [ ]:
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

df = pd.read_csv('arizona_sunrise_sunset.csv')

# Parse ISO8601 timestamps, then convert to minutes-since-midnight in local time
def to_minutes(series):
    dt = pd.to_datetime(series, format='ISO8601', utc=True).dt.tz_convert('America/Phoenix')
    return dt.dt.hour * 60 + dt.dt.minute + dt.dt.second / 60

df['sunrise_min'] = to_minutes(df['sunrise'])
df['sunset_min']  = to_minutes(df['sunset'])

# Reference times in minutes since midnight (Arizona local)
dawn_min = 5 * 60 + 1   # 05:01
dusk_min = 20 * 60 + 4  # 20:04

df['sunrise_diff'] = df['sunrise_min'] - dawn_min
df['sunset_diff']  = df['sunset_min']  - dusk_min

# Label each point for the heatmap x-axis
df['point'] = df['lat'].round(2).astype(str) + ',' + df['lon'].round(2).astype(str)

heatmap_data = df.set_index('point')[['sunrise_diff', 'sunset_diff']].T
heatmap_data.index = ['Sunrise vs Dawn (05:01)', 'Sunset vs Dusk (20:04)']

plt.figure(figsize=(20, 3))
sns.heatmap(
    heatmap_data,
    cmap='RdYlGn_r',
    center=0,
    annot=True,
    fmt='.1f',
    cbar_kws={'label': 'Minutes difference'},
    linewidths=0.3,
)
plt.title('Sunrise/Sunset vs Reference Times Across Arizona (minutes early/late)')
plt.xlabel('lat, lon')
plt.tight_layout()
plt.show()


In [ ]:
import folium
import branca.colormap as cm

# Re-use df and diffs from cell above
STEP = 0.5  # must match grid resolution in sunrise.py

clat = (df['lat'].min() + df['lat'].max()) / 2
clon = (df['lon'].min() + df['lon'].max()) / 2
m = folium.Map(location=[clat, clon], zoom_start=6, tiles='OpenStreetMap')

def make_layer(col, label, caption):
    vmin, vmax = df[col].min(), df[col].max()
    vcenter = 0
    # Diverging colormap: green (early) → white (on-time) → red (late)
    cmap = cm.LinearColormap(
        ['#2ecc71', '#f9f9f9', '#e74c3c'],
        vmin=vmin, vmax=vmax,
        caption=caption
    )
    layer = folium.FeatureGroup(name=label, show=(col == 'sunrise_diff'))
    for _, row in df.iterrows():
        folium.Rectangle(
            bounds=[
                [row['lat'] - STEP / 2, row['lon'] - STEP / 2],
                [row['lat'] + STEP / 2, row['lon'] + STEP / 2],
            ],
            color=None,
            fill=True,
            fill_color=cmap(row[col]),
            fill_opacity=0.65,
            tooltip=(
                f"<b>lat {row['lat']}, lon {row['lon']}</b><br>"
                f"{label}: {row[col]:+.1f} min"
            ),
        ).add_to(layer)
    return layer, cmap

sunrise_layer, sunrise_cmap = make_layer(
    'sunrise_diff', 'Sunrise vs Dawn (05:01)', 'Sunrise diff (min)'
)
sunset_layer, sunset_cmap = make_layer(
    'sunset_diff', 'Sunset vs Dusk (20:04)', 'Sunset diff (min)'
)

sunrise_layer.add_to(m)
sunset_layer.add_to(m)
sunrise_cmap.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

m.save('arizona_sunrise_heatmap.html')
m
